# 01 — Explore IUCN Red List Data

This notebook pulls data from the IUCN Red List API and does an initial exploration.

**Before running:** make sure your `.env` file (in the project root) contains:
```
IUCN_TOKEN=your_actual_token_here
```
Get a free token here: https://apiv3.iucnredlist.org/api/v3/token (takes 1-2 days to arrive).

**Goals for this notebook:**
- Pull all species + their current Red List category
- See the breakdown by taxonomic group (mammals, birds, amphibians, etc.)
- Pull historical assessments for a few species
- Save results to the SQLite database

In [1]:
import sys
sys.path.append('..')  # so we can import from src/

import os
import pandas as pd
import plotly.express as px

from src.iucn import get_assessments_by_taxonomy, get_assessments_by_name, TAXONOMIC_GROUPS
from src.db import init_db, df_to_table, query

In [2]:
from dotenv import load_dotenv

# Load .env from the project root (one level up from notebooks/)
load_dotenv('../.env')

# Verify the token was found
import os
if not os.environ.get("IUCN_TOKEN"):
    raise EnvironmentError("IUCN_TOKEN not found. Check that your .env file exists at conservation-tracker/.env and contains IUCN_TOKEN=...")
print("Token loaded successfully.")

Token loaded successfully.


In [3]:
# Set up the database (creates tables if they don't exist)
init_db()

Database ready at c:\Users\ronal\Desktop\Conservation Project\conservation-tracker\notebooks\..\db\conservation.db


## 1. Pull species by taxonomic group

We fetch assessments for each major animal class — mammals, birds, amphibians, reptiles, fish, insects.
The API returns one batch per class so we loop through all of them and combine into one DataFrame.

In [4]:
all_rows = []

for level, name in TAXONOMIC_GROUPS:
    print(f"Fetching {name}...")
    result = get_assessments_by_taxonomy(level, name)
    batch = result.get("assessments", [])
    for row in batch:
        row["tax_class"] = name   # tag which group this came from
    all_rows.extend(batch)
    print(f"  → {len(batch):,} assessments")

df_species = pd.DataFrame(all_rows)
print(f"\nTotal assessments: {len(df_species):,}")
df_species.head()

Fetching mammalia...
  → 100 assessments
Fetching aves...
  → 100 assessments
Fetching amphibia...
  → 100 assessments
Fetching reptilia...
  → 100 assessments
Fetching actinopterygii...
  → 100 assessments
Fetching insecta...
  → 100 assessments

Total assessments: 600


,year_published,latest,possibly_extinct,possibly_extinct_in_the_wild,sis_taxon_id,url,taxon_scientific_name,red_list_category_code,assessment_id,scopes,tax_class
0,2016,False,False,False,11481,https://www.iucnredlist.org/species/11481/503146,Lemmus lemmus,LC,503146,"[{'description': {'en': 'Global'}, 'code': '1'...",mammalia
1,2019,False,False,False,11797,https://www.iucnredlist.org/species/11797/503908,Lepus castroviejoi,VU,503908,"[{'description': {'en': 'Global'}, 'code': '1'...",mammalia
2,2012,False,False,False,12817,https://www.iucnredlist.org/species/12817/509971,Marmosops cracens,DD,509971,"[{'description': {'en': 'Global'}, 'code': '1'}]",mammalia
3,2016,False,False,False,12835,https://www.iucnredlist.org/species/12835/510082,Marmota marmota,LC,510082,"[{'description': {'en': 'Global'}, 'code': '1'...",mammalia
4,2019,False,False,False,13462,https://www.iucnredlist.org/species/13462/513175,Microtus felteni,LC,513175,"[{'description': {'en': 'Global'}, 'code': '1'...",mammalia


In [5]:
# Keep only the latest assessment per species to avoid duplicates
df_latest = df_species[df_species["latest"] == True].copy()
print(f"Latest assessments only: {len(df_latest):,}")

# Save raw data
df_latest.to_csv('../data/raw/iucn_species_latest.csv', index=False)
print("Saved to data/raw/iucn_species_latest.csv")

Latest assessments only: 302
Saved to data/raw/iucn_species_latest.csv


In [6]:
# Save to database
df_to_table(df_latest, 'species', if_exists='replace')
print("Saved to database table: species")

Saved to database table: species


## 2. Breakdown by Red List category

Categories from most to least threatened:
- **EX** — Extinct
- **EW** — Extinct in the Wild
- **CR** — Critically Endangered
- **EN** — Endangered
- **VU** — Vulnerable
- **NT** — Near Threatened
- **LC** — Least Concern
- **DD** — Data Deficient

In [7]:
category_order = ['EX', 'EW', 'CR', 'EN', 'VU', 'NT', 'LC', 'DD']

category_counts = (
    df_latest['red_list_category_code']
    .value_counts()
    .reindex(category_order, fill_value=0)
    .reset_index()
)
category_counts.columns = ['category', 'count']

fig = px.bar(
    category_counts,
    x='category',
    y='count',
    color='category',
    title='Species Count by Red List Category',
    labels={'count': 'Number of Species', 'category': 'Red List Category'}
)
fig.show()

## 3. Breakdown by taxonomic group (class)

Which animal groups — mammals, birds, amphibians, etc. — have the most threatened species?

In [8]:
threatened = ['CR', 'EN', 'VU']

df_threatened = df_latest[df_latest['red_list_category_code'].isin(threatened)]

by_class = (
    df_threatened
    .groupby('tax_class')['assessment_id']
    .count()
    .sort_values(ascending=False)
    .reset_index()
)
by_class.columns = ['class', 'threatened_species']

fig2 = px.bar(
    by_class,
    x='threatened_species',
    y='class',
    orientation='h',
    title='Threatened Species by Taxonomic Class (CR + EN + VU)',
    labels={'threatened_species': 'Threatened Species Count', 'class': 'Class'}
)
fig2.show()

## 4. Historical assessments — success stories

Pull the full assessment history for a few well-known species that improved status.

In [9]:
# Success stories — species with known recovery
# (genus, species) tuples
success_species = [
    ("haliaeetus", "leucocephalus"),  # Bald Eagle
    ("megaptera", "novaeangliae"),    # Humpback Whale
    ("panthera", "tigris"),           # Tiger
]

history_rows = []

for genus, species in success_species:
    print(f"Fetching history for {genus} {species}...")
    result = get_assessments_by_name(genus, species)
    for entry in result.get('assessments', []):
        entry['query_name'] = f"{genus.capitalize()} {species}"
        history_rows.append(entry)

df_history = pd.DataFrame(history_rows)
print(f"\nTotal historical records: {len(df_history)}")
df_history[['query_name', 'year_published', 'red_list_category_code']].sort_values(['query_name', 'year_published'])

Fetching history for haliaeetus leucocephalus...
Fetching history for megaptera novaeangliae...
Fetching history for panthera tigris...

Total historical records: 31


,query_name,year_published,red_list_category_code
8,Haliaeetus leucocephalus,1988,LR/lc
7,Haliaeetus leucocephalus,1994,LR/lc
6,Haliaeetus leucocephalus,2000,LR/lc
5,Haliaeetus leucocephalus,2004,LC
4,Haliaeetus leucocephalus,2008,LC
3,Haliaeetus leucocephalus,2009,LC
2,Haliaeetus leucocephalus,2012,LC
1,Haliaeetus leucocephalus,2016,LC
0,Haliaeetus leucocephalus,2024,LC
18,Megaptera novaeangliae,1965,N/A


In [10]:
threat_level = {'LC': 1, 'NT': 2, 'VU': 3, 'EN': 4, 'CR': 5, 'EW': 6, 'EX': 7}

df_history['threat_score'] = df_history['red_list_category_code'].map(threat_level)
df_history['year'] = pd.to_numeric(df_history['year_published'], errors='coerce')

fig3 = px.line(
    df_history.dropna(subset=['year', 'threat_score']),
    x='year',
    y='threat_score',
    color='query_name',
    markers=True,
    title='Red List Status Over Time — Success Stories',
    labels={'threat_score': 'Threat Level', 'year': 'Year', 'query_name': 'Species'}
)
fig3.update_yaxes(tickvals=list(threat_level.values()), ticktext=list(threat_level.keys()))
fig3.show()

In [11]:
# Save history to DB
df_to_table(df_history, 'assessments', if_exists='replace')
print("Saved to database table: assessments")

Saved to database table: assessments


## 5. Quick DB check

Verify the data landed in SQLite correctly.

In [12]:
query("SELECT red_list_category_code as category, COUNT(*) as count FROM species GROUP BY red_list_category_code ORDER BY count DESC")

,category,count
0,LC,139
1,DD,70
2,EN,36
3,VU,22
4,CR,22
5,NT,8
6,EX,3
7,RE,1
8,EW,1
